# Frame My Lane as an ML Task

[Open in Colab](https://colab.research.google.com/github/vibhanshu-s/FlyrankAI-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This notebook builds on my Week 1 question: which pages should an editor review first? I am using the starter data to make the task and its limits more concrete. The explanations are my working framing, and I can revise them as I learn more.

## 1. My lane as an ML task (type)

My lane is **Refresh / Content Opportunity Scoring**. I see this mainly as a **ranking task**: I want to put pages in a useful order for an editor to review. A score would help create that order, but a high score would not mean a refresh is guaranteed to work.

A classification model could eventually estimate whether a page is likely to lose impressions, and I could use that estimate as one part of the ranking. I am not training that model here. First I need a clear target and a fair way to check the result.

The output would be a list of up to 20 pages per client, with a short reason for each suggestion. An editor could check the actual content, decide whether it needs an update, or keep monitoring it. A wrong suggestion wastes review time and may lead to unnecessary changes; a missed page could delay attention to a real problem.

Here is how I would connect the project to the ML loop:

| Step | What it means for this project |
|---|---|
| Frame the decision | Choose which pages an editor should review first. |
| Prepare the data | Keep one row per page and check coverage, missing values, and time windows. |
| Build a starting point | Rank pages by impressions before trying a more detailed score. |
| Learn and evaluate | Try a model only after defining a future outcome; compare it on later periods and clients kept aside. |
| Use the result | Give the editor a short review list with reasons, rather than automatically changing pages. |
| Review feedback | Check which suggestions were useful and revisit the method. Editor feedback is not present in this dataset yet. |

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

MIN_IMPRESSIONS = 100
TOP_K = 20
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data/raw/content_refresh_anonymized.csv").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run this notebook inside a repository checkout containing the starter CSV.")
starter = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")
assert starter["content_id"].is_unique, "Check the page grain before continuing."
assert not starter[["content_id", "client_id"]].isna().any().any()

# Keep pages with some measured demand, including both declining and other pages.
lane = starter.loc[starter["impressions_90d"].ge(MIN_IMPRESSIONS)].copy()
assert not lane.empty
print(f"Starter: {len(starter):,} pages, {starter.client_id.nunique()} clients, {len(starter.columns)} columns.")
print(f"Lane slice: {len(lane):,} pages across {lane.client_id.nunique()} clients.")
print(f"Filter: at least {MIN_IMPRESSIONS} impressions over 90 days; no filter on decline.")

Starter: 30,000 pages, 32 clients, 44 columns.
Lane slice: 22,006 pages across 30 clients.
Filter: at least 100 impressions over 90 days; no filter on decline.


## 2. Target or proxy

For this notebook, I am using **recent impression decline as a temporary proxy** for a page that may deserve review. The column `recent_decline_proxy` is 1 when the supplied `trend_direction` is `down`, and 0 for a measured comparison that is not down. The dataset defines down as a fall of more than 20% from the previous 30 days to the latest 30 days.

If the previous period had zero impressions, a percentage decline cannot be assessed. I leave those labels missing instead of calling them successful or unsuccessful pages. That is a small change from the starter pipeline's simple down-versus-everything-else label. A non-declining page can still need editing, and a declining page may not benefit from it.

This proxy is calculated from observed measurements using a rule. It is not an editor's judgment, a future outcome, or evidence that refreshing a page will help. I use it below to make the metric concrete, not to claim I have predicted anything.

For a later model, I would prefer **`future_decline_target`**: 1 if next-30-day impressions fall by more than 20% compared with the preceding 30 days, otherwise 0, for pages with at least 100 impressions in the preceding period and complete search coverage. Missing coverage would stay missing. I would calculate this from daily warehouse measurements after the decision date, using only earlier data as inputs. The starter snapshot cannot fill that target, so its column stays empty here.

I would keep `trend_direction`, `trend_pct`, the 30-day impression fields used to make the proxy, and the target columns out of model inputs. The 90-day totals also overlap the current proxy's period, so this snapshot cannot give me an honest future-prediction test just by splitting clients.

In [2]:
has_comparison = lane["impressions_prev_30d"].gt(0)
assert lane.loc[has_comparison, "trend_direction"].isin(["down", "stable", "up"]).all()
lane["recent_decline_proxy"] = pd.Series(pd.NA, index=lane.index, dtype="Int64")
lane.loc[has_comparison, "recent_decline_proxy"] = (
    lane.loc[has_comparison, "trend_direction"].eq("down").astype("int64")
)
lane["future_decline_target"] = pd.Series(pd.NA, index=lane.index, dtype="Int64")

counts = lane["recent_decline_proxy"].value_counts().reindex([0, 1], fill_value=0)
target_summary = pd.DataFrame({
    "meaning": ["Measured comparison, not down", "Measured comparison, down", "No previous impressions"],
    "pages": [int(counts.loc[0]), int(counts.loc[1]), int(lane.recent_decline_proxy.isna().sum())],
}, index=["proxy = 0", "proxy = 1", "proxy missing"])
display(target_summary)
print("Future target: all values are missing because there is no later outcome window here.")
assert lane.future_decline_target.isna().all()

,meaning,pages
proxy = 0,"Measured comparison, not down",8606
proxy = 1,"Measured comparison, down",13152
proxy missing,No previous impressions,248


Future target: all values are missing because there is no later outcome window here.


## 3. Success metric

My main metric is **precision@20**: among the 20 pages selected for a client, how many have the outcome I am looking for? For example, 12 labelled declines in a list of 20 would give 60%. That is an illustration, not a result from the data.

For today's check, the outcome is the recent-decline proxy. I rank pages by 90-day impressions, then calculate the fraction with that proxy in the top 20. I average the results across clients with full lists so a large client does not dominate. Clients with fewer than 20 labelled pages are reported separately. Pages with missing labels are excluded from this diagnostic, and I report how many that leaves out. I also show the average decline rate in the same clients' eligible pools for context.

**What I would count as good later:** Before training, I am setting a working goal of at least **5 percentage points above this simple ranking method**, using the future target on later periods and clients kept aside, with the same review limit and eligibility rules. I would check uncertainty across clients and time periods and review the reasons with an editor before calling the difference useful. That is a provisional project goal, not a score I have achieved. Any future baseline will need to be measured again on those future windows.

This metric fits a limited review budget better than overall accuracy. It still cannot measure the value of an edit; for that I would need editor feedback and evidence about what happened after a change.

In [3]:
measured = lane.loc[lane["recent_decline_proxy"].notna()].copy()
queue = (
    measured.sort_values(["client_id", "impressions_90d", "content_id"],
                         ascending=[True, False, True])
    .groupby("client_id", sort=False).head(TOP_K)
)
# Pseudonymous IDs only group pages or break ties; they are not predictive inputs.
client_scores = queue.groupby("client_id")["recent_decline_proxy"].agg(["mean", "size"])
full = client_scores.loc[client_scores["size"].eq(TOP_K)]
short = client_scores.loc[client_scores["size"].lt(TOP_K)]
assert not full.empty
pool_rates = measured.groupby("client_id")["recent_decline_proxy"].mean()

print(f"Labelled pages used: {len(measured):,} / {len(lane):,}.")
print(f"Pages left out because the proxy is missing: {lane.recent_decline_proxy.isna().sum():,}.")
print(f"Clients with no labelled pages: {lane.client_id.nunique() - len(client_scores)}.")
print(f"Mean precision@{TOP_K}: {100 * full['mean'].mean():.2f}% across {len(full)} full lists.")
print(f"Mean eligible-pool decline rate for those same clients: {100 * pool_rates.loc[full.index].mean():.2f}%.")
print(f"Clients with shorter lists: {len(short)}.")
if not short.empty:
    print(f"Mean decline share in shorter lists: {100 * short['mean'].mean():.2f}%.")
print("These are descriptions of recent decline, not model validation or refresh success.")

Labelled pages used: 21,758 / 22,006.
Pages left out because the proxy is missing: 248.
Clients with no labelled pages: 0.
Mean precision@20: 47.92% across 24 full lists.
Mean eligible-pool decline rate for those same clients: 60.33%.
Clients with shorter lists: 6.
Mean decline share in shorter lists: 41.94%.
These are descriptions of recent decline, not model validation or refresh success.


**My starting result:** The simple ranking gives **47.92% mean precision@20** across 24 clients with full lists. The same clients' eligible pools have a mean recent-decline rate of **60.33%**, so ordering by impressions alone does not concentrate declining pages in this check. That gives me a useful comparison to investigate, but it does not show that a model would improve it.

These numbers differ slightly from Week 1 because I now leave the 248 pages with no previous-period impressions unlabelled and exclude them from this diagnostic. There are also six clients with shorter lists, reported separately above.

## 4. The unit of analysis, as a real dataframe

**One row = one page belonging to one client, at the starter snapshot.** I kept pages with at least 100 impressions over 90 days. I did not select only declining pages, because a ranking needs other pages to compare with them.

The small dataframe below shows actual rows from that slice. The page and client IDs are pseudonyms used to identify and group rows. The two target columns sit beside the measurements so the planned structure is easy to see, but they would not be model inputs.

I treat position zero as missing measurement and keep missing word counts visible. CTR is already a percentage: 0.76 means 0.76%, not 76%. These measurements are useful for exploration; future prediction would require reconstructing them from an earlier window.

In [4]:
lane["avg_position_observed"] = lane["avg_position"].mask(lane["avg_position"].eq(0))
lane["has_position"] = lane["avg_position"].gt(0)
lane["has_word_count"] = lane["word_count"].notna()
view_columns = [
    "content_id", "client_id", "impressions_90d", "ctr", "avg_position_observed",
    "days_since_last_update", "word_count", "recent_decline_proxy", "future_decline_target",
]
page_frame = lane[view_columns].copy()
assert page_frame["content_id"].is_unique
print(f"Page-level dataframe: {len(page_frame):,} rows and {len(page_frame.columns)} displayed columns.")
print("First five actual pages; IDs are pseudonymized and the future target is unobserved.")
display(page_frame.head(5))

Page-level dataframe: 22,006 rows and 9 displayed columns.
First five actual pages; IDs are pseudonymized and the future target is unobserved.


,content_id,client_id,impressions_90d,ctr,avg_position_observed,days_since_last_update,word_count,recent_decline_proxy,future_decline_target
0,content_304f48230142,client_f369cb89fc,3803,0.76,10.6,20,3221.0,1,<NA>
1,content_a1fb4e703a9e,client_4e07408562,15320,0.05,20.3,25,2481.0,1,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,12581,0.09,36.5,20,3515.0,1,<NA>
3,content_331d6c4de07b,client_19581e27de,11751,0.49,6.2,22,NaN,0,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,19140,0.13,44.0,14,2803.0,1,<NA>


## 5. Why ML beats a fixed rule here

I have not shown that ML beats a rule yet. A rule such as 'review older pages with search demand' is easy to explain and may be enough. My Week 1 check found only 17 pages matching that rule, compared with 13,152 pages with recent decline and at least 100 impressions. That makes me interested in other ways to narrow the review list, but it does not prove that the rule is wrong.

Pages can differ in demand, position, engagement, and update age. Those signals may interact: an older page might still be doing well, while a recently updated page might be losing visibility. A model could help if those combinations contain useful patterns that a simple rule misses. The summary below shows variation and missing measurements, which are things I need to understand before modelling; variation by itself does not prove predictive signal.

I will compare a simple method first, keep the future outcome separate from the inputs, and only move to ML if it improves the held-out result enough to matter. I would check history coverage for each client, keep the warehouse's final month for the final test, and treat unavailable tracking as missing rather than zero. If a rule performs just as well, I will use the rule.

The final action stays the same: help an editor choose what to inspect next. Even a good decline predictor would not tell me that a refresh caused recovery or should be applied automatically.

In [5]:
signals = ["impressions_90d", "avg_position_observed", "days_since_last_update", "word_count"]
signal_summary = pd.DataFrame({
    "minimum": lane[signals].min(),
    "median": lane[signals].median(),
    "maximum": lane[signals].max(),
    "missing_pct": lane[signals].isna().mean().mul(100),
})
display(signal_summary.round(2))
print("Missing values remain missing; these are exploratory signals, not fitted model inputs.")

,minimum,median,maximum,missing_pct
impressions_90d,100.0,1704.5,517715.0,0.00
avg_position_observed,0.1,12.3,88.9,0.00
days_since_last_update,4.0,22.0,313.0,0.00
word_count,675.0,2927.0,9546.0,29.79


Missing values remain missing; these are exploratory signals, not fitted model inputs.


## 6. Self-check

- [x] I named the ranking task and linked it to an editor's review decision.
- [x] I mapped the project onto the ML loop.
- [x] I explained the current proxy, missing labels, and the future target I would prefer.
- [x] I chose precision@20, defined a working success goal, and calculated a simple starting comparison.
- [x] I showed real page-level rows from the lane slice without private names, URLs, or queries.
- [x] I explained why ML might help without claiming it already beats a rule.
- [x] I kept observed decline separate from the benefit of refreshing a page.
- [x] The notebook runs top to bottom and has saved outputs.
- [x] The notebook is saved and committed under `work/notebooks/`.

My repository: [FlyrankAI-Internship](https://github.com/vibhanshu-s/FlyrankAI-Internship).